In [ ]:
# importing libraries
import pandas as pd
import numpy as np
import ast

In [ ]:
# reading csv files into dataframes
movies = pd.read_csv('tmdb_5000_movies.csv')
credits = pd.read_csv('tmdb_5000_credits.csv')

In [ ]:
# merging both the data frames
movies = movies.merge(credits, on = 'title')

In [ ]:
# removing unwanted columns
movies = movies[['movie_id', 'title', 'overview', 'genres', 'keywords', 'cast', 'crew']]

In [ ]:
# identifying rows with missing values
movies.isnull().sum()

In [ ]:
# removing rows with missing values
movies.dropna(inplace=True)

In [ ]:
# identifying duplicate rows
movies.duplicated().sum()

In [ ]:
# cleaning genres and keywords column

def convert(obj):
  L = []
  for i in ast.literal_eval(obj):
    L.append(i['name'])
  return L

movies['genres'] = movies['genres'].apply(convert)
movies['keywords'] = movies['keywords'].apply(convert)

In [ ]:
# cleaning cast column

def convert3(obj):
  L = []
  for i in ast.literal_eval(obj):
    if len(L) < 3:
      L.append(i['name'])
    else:
      break
  return L

movies['cast'] = movies['cast'].apply(convert3)

In [ ]:
# cleaning crew column

def fetch_director(obj):
  L = []
  for i in ast.literal_eval(obj):
    if i['job'] == 'Director':
      L.append(i['name'])
      break
  return L

movies['crew'] = movies['crew'].apply(fetch_director)

In [ ]:
# converting overview into a list of words
movies['overview'] = movies['overview'].apply(lambda x: x.split())

In [ ]:
# removing whitespaces
movies['genres'] = movies['genres'].apply(lambda x: [i.replace(" ", "") for i in x])
movies['keywords'] = movies['keywords'].apply(lambda x: [i.replace(" ", "") for i in x])
movies['cast'] = movies['cast'].apply(lambda x: [i.replace(" ", "") for i in x])
movies['crew'] = movies['crew'].apply(lambda x: [i.replace(" ", "") for i in x])

In [ ]:
# concatenanting columns into a single column
movies['tags'] = movies['overview'] + movies['genres'] + movies['keywords'] + movies['cast'] + movies['crew']

In [ ]:
# creating new dataframe
movies_new = movies[['movie_id', 'title', 'tags']]

In [ ]:
#  converting lists of tags into string
movies_new['tags'] = movies_new['tags'].apply(lambda x: " ".join(x))

In [ ]:
# converting into lowercase
movies_new['tags'] = movies_new['tags'].apply(lambda x: x.lower())

In [ ]:
# stemming of words
import nltk
from nltk.stem.porter import PorterStemmer

ps = PorterStemmer()

def stem(text):
  y = []
  for i in text.split():
    y.append(ps.stem(i))
  return " ".join(y)

movies_new['tags'] = movies_new['tags'].apply(stem)

In [ ]:
# removing stop words and converting into vectors using TF-IDF
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(max_features=5000, stop_words='english')
vectors = tfidf.fit_transform(movies_new['tags'])

In [ ]:
# recommend function
def recommend(movie):
  movie_index = movies_new[movies_new['title'] == movie].index[0]
  distances = similarity[movie_index]
  movies_list = sorted(list(enumerate(distances)), reverse=True, key=lambda x: x[1])[1:6]

  for i in movies_list:
    print(movies_new.iloc[i[0]].title)

In [ ]:
# creating pickle files
import pickle

pickle.dump(movies_new.to_dict(), open('movies_dict.pkl', 'wb'))
pickle.dump(vectors, open('vectors.pkl', 'wb'))